# Notebook 8.1: Portfolio-Level Backtesting with bt

## Financial Trading with Python, 2nd Edition

---

Throughout Chapters 5, 6, and 7, every backtest used the same shortcut: daily portfolio return equals the sum of yesterday's weights times today's asset returns. That calculation is clean, fast, and useful for learning metrics. It is also incomplete. It assumes you can rebalance at exactly the closing price with no friction, that uninvested cash earns nothing, and that the only thing that matters is the signal. Real trading does not work that way.

This notebook replaces the shortcut with a proper portfolio-level backtest using `bt`, a backtesting framework built on `ffn` (Financial Functions for Python). If you are wondering why `ffn` is appearing now when we spent all of Chapter 7 building metrics from scratch, the answer is straightforward: `ffn` is not our evaluation layer. It comes along as `bt`'s dependency and handles some of `bt`'s internal calculations. We are not switching to it for performance measurement. Our Chapter 7 toolkit remains the authoritative way we evaluate any strategy in this book. We will use `bt`'s built-in summary as a sanity check, then hand the return series to our own toolkit, exactly as we did in Notebooks 7.1 and 7.2. The reason we chose `bt` is not `ffn`. It is that `bt` accepts a plain DataFrame of close prices and a pre-calculated weight DataFrame directly, which means we can plug in the signals we already built without rewriting the strategy logic from scratch.

We use the same 50/200-day SMA crossover on SPY and BIL that served as the teaching vehicle in Notebook 7.1. The strategy is still deliberately basic. The goal here is to understand what a portfolio-level backtester adds, and to quantify exactly how much the simplified approach was hiding.

This notebook covers:

- Installing and pinning `bt` and `ffn` for reproducibility
- Understanding the bt Algo stack: how `RunDaily`, `WeighTarget`, and `Rebalance` work together
- Running the SMA crossover with no costs and comparing the result to the Chapter 7 vectorized calculation
- Understanding why the two approaches produce different numbers even with identical signals
- Introducing commission and slippage step by step and measuring the performance delta at each stage
- Extracting the return series from bt and evaluating it with the Chapter 7 toolkit
- Building a three-way comparison: Chapter 7 vectorized vs. bt no-cost vs. bt with realistic costs

## Section 1: Setup and Installation

### 1.1 Installing and Importing Libraries

We pin `bt` and `ffn` to specific versions to ensure the notebook runs consistently in Google Colab. Library APIs shift between releases, and a version mismatch is one of the more frustrating debugging experiences when you return to a notebook months later.

In [1]:
# --- Install pinned versions ---
# bt is the backtesting framework; ffn is its dependency and handles
# internal performance calculations. We pin both to avoid API drift.
!pip install bt==1.1.5 ffn==1.1.5 -q

# The warnings can be ignored.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.4/297.4 kB 6.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import bt
import ffn
import warnings

warnings.filterwarnings('ignore')
sns.set_theme()

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

# Confirm versions
print(f"bt version:  {bt.__version__}")
print(f"ffn version: {ffn.__version__}")

bt version:  1.1.5
ffn version: 1.1.5


## Section 2: Data Preparation

### 2.1 Loading the Price Data

We load the same case study price data used throughout the book. For the SMA crossover strategy we only need SPY and BIL — the same two assets used in Notebook 7.1. Keeping the data identical means any differences we find between the bt result and the Chapter 7 result come from the backtester, not the inputs.

In [3]:
# --- Load case study data ---
df_prices = pd.read_parquet('case_study_prices.parquet')

# We only need SPY and BIL for the crossover strategy
df_prices = df_prices[['SPY', 'BIL']].copy()

print(f"Data loaded: {df_prices.shape[0]} trading days")
print(f"Date range: {df_prices.index[0].strftime('%Y-%m-%d')} to "
      f"{df_prices.index[-1].strftime('%Y-%m-%d')}")
print(f"\nFirst few rows:")
print(df_prices.head())

Data loaded: 3771 trading days
Date range: 2011-01-04 to 2025-12-31

First few rows:
               SPY     BIL
Date                      
2011-01-04 97.1394 74.9744
2011-01-05 97.6442 74.9744
2011-01-06 97.4530 74.9744
2011-01-07 97.2618 74.9580
2011-01-10 97.1394 74.9744


### 2.2 Building the Weight DataFrame

bt accepts a pre-calculated weight DataFrame directly via its `WeighTarget` algo. We build the weights exactly as we did in Notebook 7.1 — 50/200-day SMA crossover on SPY, with BIL as the cash position — trimmed to the first valid signal date so the strategy and benchmark start on equal footing.

In [4]:
# --- Calculate moving averages ---
sma_50 = df_prices['SPY'].rolling(window=50).mean()
sma_200 = df_prices['SPY'].rolling(window=200).mean()

# --- Find the first valid signal date ---
# sma_200 is NaN for the first 199 days. Starting the strategy before
# this date means sitting in BIL by default while the benchmark is
# already fully invested in SPY — an unfair comparison.
start_date = sma_200.first_valid_index()

# --- Generate signal from start_date onward ---
signal = (sma_50 > sma_200).loc[start_date:].astype(float)

# --- Build weight DataFrame ---
# bt expects weights as floats between 0 and 1,
# with column names matching the price DataFrame columns.
df_weights = pd.DataFrame(index=signal.index)
df_weights['SPY'] = signal
df_weights['BIL'] = 1 - signal

# --- Trim prices to match ---
# bt uses the price DataFrame to define the backtest period.
# Starting both from the same date ensures a fair comparison
# with the benchmark, which is also measured from start_date.
df_prices_bt = df_prices.loc[start_date:].copy()

print(f"Strategy start date: {start_date.strftime('%Y-%m-%d')}")
print(f"Weight DataFrame: {df_weights.shape[0]} rows")
print(f"Price DataFrame:  {df_prices_bt.shape[0]} rows")
print(f"\nDays long SPY: {(df_weights['SPY'] == 1).sum()} "
      f"({(df_weights['SPY'] == 1).mean():.1%})")
print(f"Days in BIL:   {(df_weights['BIL'] == 1).sum()} "
      f"({(df_weights['BIL'] == 1).mean():.1%})")
print(f"\nFirst few rows:")
print(df_weights.head())

Strategy start date: 2011-10-18
Weight DataFrame: 3572 rows
Price DataFrame:  3572 rows

Days long SPY: 2973 (83.2%)
Days in BIL:   599 (16.8%)

First few rows:
              SPY    BIL
Date                    
2011-10-18 0.0000 1.0000
2011-10-19 0.0000 1.0000
2011-10-20 0.0000 1.0000
2011-10-21 0.0000 1.0000
2011-10-24 0.0000 1.0000


## Section 3: Backtesting with bt

### 3.1 The bt Algo Stack

Before we run anything, it is worth understanding how bt thinks about a strategy. Rather than a single monolithic function, bt composes strategies from small, reusable building blocks called Algos. Each Algo is responsible for one decision. They run in sequence on every time step, and each one can pass or block execution to the next.

For our strategy we need three:

- `bt.algos.RunDaily()`: tells bt to evaluate the strategy on every trading day
- `bt.algos.WeighTarget(df_weights)`: sets the target weights from our pre-calculated DataFrame
- `bt.algos.Rebalance()`: executes trades to bring the portfolio in line with the target weights

That is the entire strategy definition. The signal logic stays in our weight DataFrame where we built it. bt just executes it.

In [5]:
# --- Define the bt strategy ---
# The Algo stack runs top to bottom on each time step.
# RunDaily: evaluate every day (as opposed to weekly, monthly, etc.)
# WeighTarget: load today's target weights from our pre-calculated DataFrame
# Rebalance: execute trades to match the target weights
strategy_no_cost = bt.Strategy(
    'SMA_Crossover_No_Cost',
    [bt.algos.RunDaily(),
     bt.algos.WeighTarget(df_weights),
     bt.algos.Rebalance()]
)

# --- Create the bt Backtest object ---
# We pass df_prices_bt (trimmed to the strategy period) so the backtest
# starts on the same date as our first valid signal — not on 2011-01-04.
# initial_capital=1000 gives us a clean index starting at 1.0 when we
# divide later. The choice of starting capital does not affect returns.
backtest_no_cost = bt.Backtest(
    strategy_no_cost,
    df_prices_bt,
    initial_capital=1000
)

# --- Run the backtest ---
res_no_cost = bt.run(backtest_no_cost)

print("Backtest complete.")
print(f"\nbt built-in summary:")
res_no_cost.display()

Backtest complete.

bt built-in summary:
Stat                 SMA_Crossover_No_Cost
-------------------  -----------------------
Start                2011-10-17
End                  2025-12-31
Risk-free rate       0.00%

Total Return         272.31%
Daily Sharpe         0.76
Daily Sortino        1.05
CAGR                 9.69%
Max Drawdown         -31.73%
Calmar Ratio         0.31

MTD                  0.07%
3m                   2.44%
6m                   9.98%
YTD                  0.84%
1Y                   0.84%
3Y (ann.)            13.56%
5Y (ann.)            11.81%
10Y (ann.)           9.18%
Since Incep. (ann.)  9.69%

Daily Sharpe         0.76
Daily Sortino        1.05
Daily Mean (ann.)    10.18%
Daily Vol (ann.)     13.46%
Daily Skew           -0.54
Daily Kurt           23.73
Best Day             9.78%
Worst Day            -10.15%

Monthly Sharpe       0.92
Monthly Sortino      1.45
Monthly Mean (ann.)  9.90%
Monthly Vol (ann.)   10.80%
Monthly Skew         -0.55
Monthly Kurt    

The bt summary gives us a complete picture in one call. Before we dig into the differences with Chapter 7, a few things to note about the output itself.

The start date shows 2011-10-17, one day before our weight DataFrame starts on 2011-10-18. This is expected: bt reports the date of the first price observation it received, not the date of the first trade. The first actual position is taken on 2011-10-18.

The numbers that matter for comparison with Chapter 7 are the CAGR, Sharpe, and max drawdown. They do not match, and the differences are not rounding. bt reports a CAGR of 9.69%, a Daily Sharpe of 0.76, and a max drawdown of -31.73%. Our Notebook 7.1 calculations gave 10.14%, 0.715, and -33.72% respectively. These gaps have specific causes that we will work through in Section 3.3.

For now, treat this output as a sanity check rather than the authoritative evaluation. The numbers are in a reasonable range — this looks like the same strategy we built in Chapter 7, not a completely different one. That is what we need to confirm before adding costs.

We will use the Chapter 7 toolkit as our authoritative evaluation layer. The bt built-in summary is useful for a quick read, but we will extract the return series and run it through `calculate_performance_metrics()` before drawing any conclusions.

### 3.2 Why bt Produces Different Results Than Chapter 7

Before we add transaction costs, we need to understand why bt and Chapter 7 give different numbers even with identical signals and data. There are three reasons.

**Execution timing.** In Chapter 7, we shifted weights by one day manually: today's return uses yesterday's weights. bt handles execution differently — when it receives today's weight, it executes the rebalance at today's close, meaning the new weights take effect from the next day's open. The net result is similar but not identical, because bt also accounts for the fact that the portfolio needs to be valued before rebalancing occurs each day.

**Uninvested cash treatment.** In Chapter 7's simplified calculation, when the strategy holds BIL, it earns BIL's daily return directly. bt tracks the portfolio at the asset level and handles cash more precisely, which can produce small differences in how BIL returns accumulate.

**CAGR annualization.** Our Chapter 7 CAGR formula uses trading days: `(1 + total_return) ** (252 / total_trading_days) - 1`. bt and ffn use actual calendar days internally: `(1 + total_return) ** (365 / calendar_days) - 1`. Over a 14-year period these two methods produce meaningfully different annualized returns from the same total return.

Let us quantify all three gaps with a direct comparison.

In [6]:
# --- Extract the return series from bt ---
# bt stores the portfolio value history in res.prices.
# We convert to daily returns for comparison with our Chapter 7 series.
#
# .prices returns a DataFrame with one column per strategy.
# We take the SMA_Crossover_No_Cost column and compute pct_change().
bt_prices = res_no_cost.prices['SMA_Crossover_No_Cost']
bt_returns = bt_prices.pct_change().dropna()
bt_returns.name = 'bt_No_Cost'

# --- Rebuild Chapter 7 return series for comparison ---
# Same logic as Notebook 7.1: lag weights by one day, apply to returns.
df_returns = df_prices_bt.pct_change().dropna()
common_idx = df_weights.index.intersection(df_returns.index)
w_lagged = df_weights.loc[common_idx].shift(1).dropna()
r_aligned = df_returns.loc[w_lagged.index]
ch7_returns = (w_lagged * r_aligned).sum(axis=1)
ch7_returns.name = 'Ch7_Vectorized'

# --- Align both series to the same dates ---
# bt may have slightly different start/end dates due to internal handling.
# We find the common date range before comparing.
common_dates = bt_returns.index.intersection(ch7_returns.index)
bt_aligned = bt_returns.loc[common_dates]
ch7_aligned = ch7_returns.loc[common_dates]

print(f"bt return series:  {len(bt_aligned)} days "
      f"({bt_aligned.index[0].strftime('%Y-%m-%d')} to "
      f"{bt_aligned.index[-1].strftime('%Y-%m-%d')})")
print(f"Ch7 return series: {len(ch7_aligned)} days "
      f"({ch7_aligned.index[0].strftime('%Y-%m-%d')} to "
      f"{ch7_aligned.index[-1].strftime('%Y-%m-%d')})")
print(f"\nCorrelation between bt and Ch7 returns: "
      f"{bt_aligned.corr(ch7_aligned):.6f}")
print(f"\nMean daily return difference (bt - Ch7): "
      f"{(bt_aligned - ch7_aligned).mean():.8f}")

bt return series:  3570 days (2011-10-20 to 2025-12-31)
Ch7 return series: 3570 days (2011-10-20 to 2025-12-31)

Correlation between bt and Ch7 returns: 0.999547

Mean daily return difference (bt - Ch7): -0.00001936


The correlation of 0.999547 tells us the two return series are nearly identical — they are tracking the same strategy with the same signal on the same data. The mean daily return difference of -0.0000194 is tiny but not zero. That small gap compounds over 3,570 trading days into the differences we saw in the summary statistics.

The two series start on 2011-10-20, two days after our weight DataFrame's start date of 2011-10-18. bt needs one day to initialize the portfolio and one day to execute the first rebalance, which is consistent with its internal execution timing. This is one of the three sources of difference we described above — bt's execution timing is slightly different from our manual one-day weight lag.

The high correlation confirms we are looking at the same strategy. The non-zero mean difference confirms the backtester is doing something our simplified calculation was not. In Section 3.3 we will quantify the full gap with a side-by-side metric comparison.

### 3.3 Comparing bt to Chapter 7

The `calculate_performance_metrics()` function we built in Notebook 7.1 is stored in `functions.py` in the book's GitHub repository. Drag and drop the file into the Colab file panel on the left, then run the cell below to import it.

In [8]:
# --- Import the Chapter 7 toolkit ---
# Drag and drop functions.py into the Colab file panel before running this cell.
from functions import calculate_performance_metrics

In [9]:
# --- Run both series through the Chapter 7 toolkit ---
# We import the calculate_performance_metrics function from Notebook 7.1.
# If you do not have it available, the full function definition is in
# Notebook 7.1 Section 7. Copy and paste it into a cell above this one.
#
# We use bt_aligned and ch7_aligned — both trimmed to the same dates —
# so the comparison is perfectly apples-to-apples.

# SPY benchmark over the same period
spy_returns = df_returns['SPY'].loc[common_dates]
spy_returns.name = 'SPY_Buy_Hold'

# --- Calculate metrics for all three series ---
bt_metrics = calculate_performance_metrics(
    bt_aligned,
    benchmark_returns=spy_returns,
    risk_free_rate=0.0
)

ch7_metrics = calculate_performance_metrics(
    ch7_aligned,
    benchmark_returns=spy_returns,
    risk_free_rate=0.0
)

spy_metrics = calculate_performance_metrics(
    spy_returns,
    risk_free_rate=0.0
)

# --- Build comparison table ---
df_comparison = pd.DataFrame({
    'Ch7 Vectorized': ch7_metrics,
    'bt No Cost': bt_metrics,
    'SPY Buy & Hold': spy_metrics
})

print("Performance Comparison: Chapter 7 Vectorized vs bt No Cost")
print("=" * 65)
print(df_comparison.to_string())

Performance Comparison: Chapter 7 Vectorized vs bt No Cost
                        Ch7 Vectorized bt No Cost SPY Buy & Hold
Total Return                   293.23%    272.31%        625.35%
Ann. Return (CAGR)              10.15%      9.72%         15.01%
Ann. Volatility                 14.19%     13.46%         16.83%
Downside Deviation              10.22%      9.71%         11.88%
Max Drawdown                   -33.72%    -31.73%        -33.72%
Max DD Duration (days)             470        466            488
Ulcer Index                      7.672      7.252          6.266
Sharpe Ratio                     0.715      0.722          0.892
Sortino Ratio                    0.993      1.002          1.264
Calmar Ratio                     0.301      0.306          0.445
Ulcer Performance Index          1.323      1.341          2.396
Win Rate (monthly)               68.4%      69.2%          70.8%
Profit Factor                     2.03       2.03           2.20
Avg Win/Loss                   

The table tells a clear story. Let us work through it column by column.

**Total return and CAGR.** bt no-cost returned 272.31% versus Chapter 7's 293.23%, a gap of nearly 21 percentage points over the full period. Annualized, that is 9.72% versus 10.15%. Both calculations use the same signal on the same data with zero transaction costs. The difference comes entirely from execution timing. Chapter 7 applied yesterday's weights to today's returns using a simple shift. bt executes rebalances at the close of the signal day and prices the portfolio before executing, which means the entry and exit timing differs slightly on days where the signal changes. On a strategy that changes signal 30-40 times over 14 years, those timing differences compound.

**Volatility and drawdown.** bt reports lower volatility (13.46% vs 14.19%) and a shallower maximum drawdown (-31.73% vs -33.72%). This is consistent with the timing difference: bt's slightly different entry and exit dates mean it occasionally avoids the first day of a large move in either direction.

**Sharpe ratio.** Despite the lower total return, bt's Sharpe of 0.722 is marginally higher than Chapter 7's 0.715. The volatility reduction outweighs the return reduction on a risk-adjusted basis. The difference is small enough to be within rounding error of methodology choices.

**The SPY benchmark.** SPY shows a total return of 625.35% and CAGR of 15.01% over this common date window, slightly higher than the 616.77% and 14.91% we saw in Notebook 7.1. The common date range starts on 2011-10-20 rather than 2011-10-19, and that one-day difference captures a period where SPY was rising, shifting the benchmark slightly upward.

**The bottom line.** The simplified Chapter 7 vectorized calculation overstated the strategy's total return by about 21 percentage points relative to bt's more precise execution model. That is not noise. It is a real difference that matters when evaluating a strategy. This is exactly the gap we promised to quantify when we first introduced the simplified approach in Chapter 7.

We have not yet added transaction costs. That comes next, and it will push the numbers further apart.

### 3.4 Introducing Transaction Costs

Real trading is never free. Every time the strategy rebalances — moving from SPY to BIL or back — it incurs costs. We introduce them in two steps: commission first, then slippage. Separating them makes the performance impact of each visible.

**Commission** is the fee charged by your broker per trade. For most retail brokers trading ETFs today, commissions are effectively zero. But institutional traders, managed accounts, and strategies trading less liquid instruments still face meaningful commission costs. More importantly, even a small commission per trade compounds significantly over hundreds of rebalances. We model commission as a percentage of the trade value.

bt handles commission through a commission function passed to the `Backtest` object. The function receives the quantity traded and the price, and returns the commission amount. We use a simple percentage-of-trade-value model.

In [10]:
# --- Define a commission function ---
# bt's commission function receives:
#   quantity: number of units traded (positive = buy, negative = sell)
#   price: price per unit at execution
# It must return the commission amount in dollars.
#
# We model commission as a flat percentage of trade value.
# 10 bps (0.10%) is a conservative estimate for institutional ETF trading.
# Retail traders with zero-commission brokers can set this to 0.
COMMISSION_PCT = 0.0010  # 10 basis points

def commission_func(quantity, price):
    """
    Calculate commission as a percentage of trade value.
    Applied to every buy and sell that bt executes.
    """
    return abs(quantity) * price * COMMISSION_PCT

# --- Run backtest with commission ---
strategy_commission = bt.Strategy(
    'SMA_Crossover_Commission',
    [bt.algos.RunDaily(),
     bt.algos.WeighTarget(df_weights),
     bt.algos.Rebalance()]
)

backtest_commission = bt.Backtest(
    strategy_commission,
    df_prices_bt,
    initial_capital=1000,
    commissions=commission_func
)

res_commission = bt.run(backtest_commission)

# --- Extract returns and run through Chapter 7 toolkit ---
bt_commission_prices = res_commission.prices['SMA_Crossover_Commission']
bt_commission_returns = bt_commission_prices.pct_change().dropna()
bt_commission_returns.name = 'bt_Commission'
bt_commission_aligned = bt_commission_returns.loc[common_dates]

commission_metrics = calculate_performance_metrics(
    bt_commission_aligned,
    benchmark_returns=spy_returns,
    risk_free_rate=0.0
)

print("Performance Delta: No Cost vs With Commission (10 bps)")
print("=" * 55)
df_delta = pd.DataFrame({
    'bt No Cost': bt_metrics,
    'bt Commission': commission_metrics
})
print(df_delta.to_string())

Performance Delta: No Cost vs With Commission (10 bps)
                        bt No Cost bt Commission
Total Return               272.31%       268.60%
Ann. Return (CAGR)           9.72%         9.65%
Ann. Volatility             13.46%        13.51%
Downside Deviation           9.71%         9.74%
Max Drawdown               -31.73%       -31.87%
Max DD Duration (days)         466           466
Ulcer Index                  7.252         7.395
Sharpe Ratio                 0.722         0.714
Sortino Ratio                1.002         0.991
Calmar Ratio                 0.306         0.303
Ulcer Performance Index      1.341         1.304
Win Rate (monthly)           69.2%         68.6%
Profit Factor                 2.03          2.01
Avg Win/Loss                  0.90          0.92
Information Ratio           -0.573        -0.580


Ten basis points of commission on every rebalance costs the strategy 3.71 percentage points of total return over the full period (272.31% down to 268.60%), or about 7 basis points of annualized CAGR (9.72% down to 9.65%). The Sharpe ratio drops from 0.722 to 0.714.

That might sound small, but consider what is happening: the SMA crossover strategy changes signal roughly 30-40 times over 14 years. Each signal change involves selling one asset and buying another, two trades. At 10 bps per trade, each full rebalance costs 20 bps of portfolio value. Spread across a long holding period between rebalances, the per-day cost is tiny. But it is never zero, and it compounds.

The key insight is in the volatility row: commission slightly increases measured volatility (13.46% to 13.51%). This is because the commission is deducted on rebalance days, creating small negative return shocks on those specific days. The drawdown deepens slightly for the same reason.

If your strategy rebalanced daily instead of holding for weeks or months, the commission impact would be catastrophic. A strategy that turns over its entire portfolio every day at 10 bps would lose roughly 25% of its value annually to commissions alone. The SMA crossover's long holding periods are one of its few structural advantages. They keep transaction costs manageable.

Now let us add slippage.

### 3.5 Introducing Slippage

### 3.5 Introducing Slippage

Slippage is the difference between the price you expected to trade at and the price you actually got. When you place a market order to buy SPY, you pay the ask price, not the mid-price. When you sell, you receive the bid. That bid-ask spread is the most basic form of slippage. For liquid ETFs like SPY and BIL, the spread is typically 1-2 basis points. For less liquid instruments it can be much wider.

Unlike commission, bt does not have a built-in slippage parameter. This is not unusual — most portfolio-level backtesting frameworks treat slippage as an extension of transaction costs rather than a separate mechanism. For liquid ETFs, this is a reasonable simplification: slippage behaves identically to commission at the portfolio level. Both are frictions applied at the moment of trade, proportional to trade value. We combine them into a single commission function representing total round-trip friction.

We use 10 bps of commission plus 5 bps of slippage, for a combined 15 bps. This is a conservative but realistic estimate for institutional ETF trading. Retail traders using zero-commission brokers would use 5 bps for slippage alone.

In [12]:
# --- Model combined costs: commission + slippage ---
# bt does not have a built-in slippage parameter. For liquid ETFs,
# slippage (the bid-ask spread cost) behaves identically to commission:
# it is a friction applied at the moment of trade, proportional to
# trade value. We combine both into a single commission function.
#
# 10 bps commission + 5 bps slippage = 15 bps total round-trip friction.
# This is a conservative but realistic estimate for institutional
# ETF trading. Retail traders with zero-commission brokers would
# use 5 bps (slippage only).
TOTAL_COST_PCT = 0.0015  # 15 basis points combined

def commission_with_slippage(quantity, price):
    """
    Combined commission and slippage as a percentage of trade value.
    For liquid ETFs, slippage behaves identically to commission at
    the portfolio level, so we model them together.
    """
    return abs(quantity) * price * TOTAL_COST_PCT

strategy_costs = bt.Strategy(
    'SMA_Crossover_With_Costs',
    [bt.algos.RunDaily(),
     bt.algos.WeighTarget(df_weights),
     bt.algos.Rebalance()]
)

backtest_costs = bt.Backtest(
    strategy_costs,
    df_prices_bt,
    initial_capital=1000,
    commissions=commission_with_slippage
)

res_costs = bt.run(backtest_costs)

# --- Extract returns ---
bt_costs_prices = res_costs.prices['SMA_Crossover_With_Costs']
bt_costs_returns = bt_costs_prices.pct_change().dropna()
bt_costs_returns.name = 'bt_With_Costs'
bt_costs_aligned = bt_costs_returns.loc[common_dates]

costs_metrics = calculate_performance_metrics(
    bt_costs_aligned,
    benchmark_returns=spy_returns,
    risk_free_rate=0.0
)

print("Performance Delta: Commission Only vs Commission + Slippage (combined)")
print("=" * 65)
df_delta2 = pd.DataFrame({
    'bt Commission': commission_metrics,
    'bt With Costs': costs_metrics
})
print(df_delta2.to_string())

Performance Delta: Commission Only vs Commission + Slippage (combined)
                        bt Commission bt With Costs
Total Return                  268.60%       249.38%
Ann. Return (CAGR)              9.65%         9.23%
Ann. Volatility                13.51%        13.49%
Downside Deviation              9.74%         9.73%
Max Drawdown                  -31.87%       -32.98%
Max DD Duration (days)            466           503
Ulcer Index                     7.395         7.685
Sharpe Ratio                    0.714         0.684
Sortino Ratio                   0.991         0.949
Calmar Ratio                    0.303         0.280
Ulcer Performance Index         1.304         1.201
Win Rate (monthly)              68.6%         69.2%
Profit Factor                    2.01          1.97
Avg Win/Loss                     0.92          0.88
Information Ratio              -0.580        -0.621


Adding the slippage component brings total costs to 15 bps per rebalance. The combined impact is now visible: total return drops from 268.60% to 249.38%, annualized CAGR falls from 9.65% to 9.23%, and the Sharpe ratio drops from 0.714 to 0.684. The max drawdown deepens from -31.87% to -32.98% and the max drawdown duration extends from 466 to 503 days.

That additional 5 bps of slippage cost the strategy 19 percentage points of total return over 14 years — more than the 3.71 points that commission alone cost. This is counterintuitive until you remember that slippage applies to the full trade value on every rebalance, and the SMA crossover's trades are large: moving from 100% SPY to 100% BIL means liquidating the entire equity position. Even a small percentage of a large trade adds up.

The pattern across all three cost runs is worth pausing on. Each layer of friction — execution timing, commission, slippage — takes a bite. None of them is catastrophic on its own. Together they add up to a meaningful gap between the simplified Chapter 7 calculation and what a real trading system would have produced.

Now let us build the full three-way comparison table that closes the loop on the promise we made throughout Chapters 5 through 7.

In [13]:
# --- Build the full three-way comparison table ---
# Ch7 Vectorized: the simplified weight-lag calculation from Notebook 7.1
# bt No Cost:     proper portfolio-level backtest, zero friction
# bt With Costs:  realistic 15 bps total transaction costs per rebalance

df_three_way = pd.DataFrame({
    'Ch7 Vectorized': ch7_metrics,
    'bt No Cost':     bt_metrics,
    'bt With Costs':  costs_metrics,
    'SPY Buy & Hold': spy_metrics
})

print("Three-Way Comparison: Simplified vs Realistic Backtests")
print("=" * 75)
print(df_three_way.to_string())

# --- Quantify the gaps ---
print("\n\nGaps relative to Chapter 7 Vectorized baseline:")
print("=" * 55)

ch7_cagr = float(ch7_metrics['Ann. Return (CAGR)'].strip('%')) / 100
bt_no_cost_cagr = float(bt_metrics['Ann. Return (CAGR)'].strip('%')) / 100
bt_costs_cagr = float(costs_metrics['Ann. Return (CAGR)'].strip('%')) / 100

print(f"Execution timing gap (Ch7 vs bt no cost):  "
      f"{(bt_no_cost_cagr - ch7_cagr)*100:+.2f}% ann.")
print(f"Transaction cost gap (bt no cost vs costs):"
      f"{(bt_costs_cagr - bt_no_cost_cagr)*100:+.2f}% ann.")
print(f"Total gap (Ch7 vs bt with costs):          "
      f"{(bt_costs_cagr - ch7_cagr)*100:+.2f}% ann.")

Three-Way Comparison: Simplified vs Realistic Backtests
                        Ch7 Vectorized bt No Cost bt With Costs SPY Buy & Hold
Total Return                   293.23%    272.31%       249.38%        625.35%
Ann. Return (CAGR)              10.15%      9.72%         9.23%         15.01%
Ann. Volatility                 14.19%     13.46%        13.49%         16.83%
Downside Deviation              10.22%      9.71%         9.73%         11.88%
Max Drawdown                   -33.72%    -31.73%       -32.98%        -33.72%
Max DD Duration (days)             470        466           503            488
Ulcer Index                      7.672      7.252         7.685          6.266
Sharpe Ratio                     0.715      0.722         0.684          0.892
Sortino Ratio                    0.993      1.002         0.949          1.264
Calmar Ratio                     0.301      0.306         0.280          0.445
Ulcer Performance Index          1.323      1.341         1.201          2.

The three-way comparison makes the full picture clear.

The Chapter 7 vectorized calculation overstated the strategy's annualized return by 0.92% relative to a realistic backtest with costs. That gap breaks down into two roughly equal components: 0.43% from execution timing differences between the simplified weight-lag approach and bt's proper portfolio accounting, and 0.49% from transaction costs at 15 bps per rebalance.

Neither component is trivial. A 0.43% annual drag from execution timing alone is a reminder that even a zero-cost backtest is not a perfect simulation of reality. The simplified approach we used throughout Chapters 5 through 7 was appropriate for learning metrics and evaluating signals, but it was never meant to represent what a live system would produce.

The cost component is where the strategy's edge gets tested. The crossover's Sharpe ratio falls from 0.722 (no cost) to 0.684 (with costs), and the Calmar ratio drops from 0.306 to 0.280. These are meaningful changes. A strategy that looked competitive on a no-cost basis now looks weaker. If your edge is thin to begin with, realistic costs can eliminate it entirely. This is the practitioner rule: if your strategy's edge disappears at 10 bps of commission, you do not have an edge.

The SPY benchmark is unaffected by any of these adjustments since buy-and-hold incurs no rebalancing costs. That makes the gap between the strategy and the benchmark wider in the realistic scenario than it appeared in Chapter 7.

The win rate, profit factor, and average win/loss ratio are stable across all three columns. These are monthly metrics and are less sensitive to the daily execution differences that drive the return and Sharpe gaps. This is consistent with what we said in Notebook 7.1: trading metrics reflect the strategy's decision quality, not its execution quality.

In the next section we extract the bt return series and feed it into our Chapter 7 toolkit for the authoritative monthly evaluation.

### 3.7 Extracting Returns and Evaluating with the Chapter 7 Toolkit

The bt built-in summary is useful for a quick read but we established in Chapter 7 that our own toolkit is the authoritative evaluation layer. We have already extracted the return series in the cells above. Here we make the workflow explicit and run the final evaluation on monthly returns.

This is also where we close the loop on the promise made throughout Chapters 5 through 7: the simplified vectorized calculation was a teaching tool, not a production backtesting system. The bt with costs result is our best estimate of what this strategy would have actually produced.

In [14]:
# --- Resample all return series to monthly ---
# We evaluate on monthly returns for the authoritative comparison.
# Monthly data reduces daily noise, is less sensitive to microstructure
# effects, and is the standard frequency for institutional performance
# reporting. We compound daily returns within each month using the
# geometric product rather than summing.

def to_monthly(daily_returns):
    """Compound daily returns to monthly using geometric compounding."""
    return daily_returns.resample('ME').apply(
        lambda x: (1 + x).prod() - 1
    )

ch7_monthly = to_monthly(ch7_aligned)
bt_costs_monthly = to_monthly(bt_costs_aligned)
spy_monthly = to_monthly(spy_returns)

# --- Run Chapter 7 toolkit on monthly returns ---
# We pass periods_per_year=12 since we are working with monthly data.
ch7_monthly_metrics = calculate_performance_metrics(
    ch7_monthly,
    benchmark_returns=spy_monthly,
    risk_free_rate=0.0,
    periods_per_year=12
)

bt_costs_monthly_metrics = calculate_performance_metrics(
    bt_costs_monthly,
    benchmark_returns=spy_monthly,
    risk_free_rate=0.0,
    periods_per_year=12
)

spy_monthly_metrics = calculate_performance_metrics(
    spy_monthly,
    risk_free_rate=0.0,
    periods_per_year=12
)

# --- Final evaluation table ---
df_final = pd.DataFrame({
    'Ch7 Vectorized': ch7_monthly_metrics,
    'bt With Costs':  bt_costs_monthly_metrics,
    'SPY Buy & Hold': spy_monthly_metrics
})

print("Final Evaluation: Monthly Returns via Chapter 7 Toolkit")
print("=" * 65)
print(df_final.to_string())

Final Evaluation: Monthly Returns via Chapter 7 Toolkit
                        Ch7 Vectorized bt With Costs SPY Buy & Hold
Total Return                   293.23%       249.38%        625.35%
Ann. Return (CAGR)              10.09%         9.18%         14.92%
Ann. Volatility                 11.32%        10.77%         13.80%
Downside Deviation               7.20%         6.94%          8.45%
Max Drawdown                   -19.48%       -19.26%        -23.93%
Max DD Duration (days)              21            23             23
Ulcer Index                      6.523         6.541          5.492
Sharpe Ratio                     0.891         0.852          1.081
Sortino Ratio                    1.401         1.322          1.765
Calmar Ratio                     0.518         0.476          0.623
Ulcer Performance Index          1.546         1.403          2.716
Win Rate (monthly)               68.4%         69.2%          70.8%
Profit Factor                     2.03          1.97        

The monthly evaluation tells a cleaner story than the daily comparison.

A few things stand out immediately.

**The CAGR figures shift slightly.** The Chapter 7 monthly CAGR is 10.09% versus 10.15% on daily data. This small difference comes from the compounding convention: monthly geometric compounding is not identical to daily geometric compounding over the same period. Neither is wrong. They measure the same thing at different granularities.

**Volatility and drawdown look different from daily.** The annualized volatility drops from 14.19% (daily) to 11.32% (monthly) for the Chapter 7 series. This is expected: daily volatility captures intraday noise that washes out at the monthly level. The max drawdown also changes: -19.48% monthly versus -33.72% daily. Monthly drawdown measures peak-to-trough using month-end NAVs, so an intra-month crash that recovers before month-end does not appear. The COVID crash of March 2020 is the clearest example: on a daily basis the drawdown hit -33.72%, but if the strategy recovered any ground by month-end, the monthly drawdown is shallower. Neither number is wrong. They answer different questions.

**A note on Max DD Duration.** The function we built in Notebook 7.1 counts periods below the prior peak. With monthly data, those periods are months, not days. The 21 shown for Chapter 7 Vectorized means 21 months, roughly 1.75 years. The column label says "days" because the function was built for daily data. Keep that in mind when reading this table.

**The Sharpe ratio improves on monthly data.** The Chapter 7 monthly Sharpe of 0.891 is higher than the daily Sharpe of 0.715. This is consistent with what we said earlier: daily Sharpe ratios are noisier and tend to understate risk-adjusted performance relative to monthly. The monthly number is the more reliable figure for comparison against industry benchmarks and other strategies.

**bt with costs monthly Sharpe of 0.852** versus Chapter 7's 0.891. The realistic cost scenario costs the strategy about 4 basis points of monthly Sharpe. This is the number that matters for evaluating whether this strategy is worth trading. Against SPY's monthly Sharpe of 1.081, neither version of the crossover strategy is competitive on a risk-adjusted basis.

This is the complete picture. The simplified Chapter 7 approach overstated performance. The realistic bt backtest with costs gives us the honest number. The Chapter 7 toolkit applied to monthly returns gives us the authoritative evaluation. All three are useful. Only the last one is what you should report.

## Wrapping Up

This notebook replaced the simplified vectorized return calculation from Chapter 7 with a proper portfolio-level backtest using `bt`. The difference between the two approaches turned out to be meaningful, not cosmetic.

The simplified approach overstated annualized return by 0.92% relative to a realistic backtest with costs. That gap breaks into two components: 0.43% from execution timing differences between the weight-lag shortcut and bt's proper portfolio accounting, and 0.49% from transaction costs at 15 bps per rebalance. Neither component is catastrophic for a long-holding-period strategy like the SMA crossover. For a high-turnover strategy, either one could be fatal.

The workflow we established here is the one you carry forward for every strategy in this book:

1. Build the signal and weights in a pre-calculated DataFrame
2. Run the backtest through `bt` to get realistic execution
3. Introduce costs in layers to see each component's impact
4. Extract the return series from `bt` and evaluate with the Chapter 7 toolkit on monthly returns

The bt built-in summary is a useful sanity check, but the Chapter 7 toolkit is the authoritative evaluation layer. Always hand the return series to your own code before drawing conclusions.

One limitation is worth stating plainly: `bt` does not have a native slippage parameter. We modeled slippage by combining it with commission into a single friction rate. That is a reasonable approximation for liquid ETFs, but it hides the distinction between a fixed contractual cost (commission) and a market-impact cost that scales with trade size and liquidity (slippage). A full event-driven backtester would model these separately. We discuss this in the chapter text.

In *Notebook 8.2*, we rebuild the same analysis using VectorBT and apply it to the VIXY tail hedge strategy, our main case study. VectorBT uses a different architecture: synthesized prices from returns, sparse weight DataFrames, and a single `Portfolio.from_orders()` call. The results should be comparable to what we found here, and any differences will be worth examining.